# Atomic $S_6$ grokking on Kaggle

Settings → Accelerator → GPU. Internet нужен только для клонирования репозитория; если он отключён, добавьте исходники как Kaggle Dataset. Результаты пишутся в `/kaggle/working`, затем их нужно сохранить через **Save Version**.

In [ ]:
import os, sys, shutil, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
REPO='/kaggle/working/2026-Project-202'
if not os.path.exists(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/intsystems/2026-Project-202.git',REPO],check=True)
WORKDIR=os.path.join(REPO,'code','Grokking')
os.chdir(WORKDIR)
# Upload the current sn_atomic_grokking_colab.py as a small Kaggle Dataset.
uploaded=[]
for root, _, files in os.walk('/kaggle/input'):
    if 'sn_atomic_grokking_colab.py' in files:
        uploaded.append(os.path.join(root,'sn_atomic_grokking_colab.py'))
assert uploaded, 'Attach a Dataset containing sn_atomic_grokking_colab.py'
shutil.copy2(uploaded[0], os.path.join(WORKDIR,'sn_atomic_grokking_colab.py'))
subprocess.run([sys.executable,'-m','pip','install','-q','einops','pandas','tqdm'],check=True)

In [ ]:
import importlib, sn_atomic_grokking_colab
importlib.reload(sn_atomic_grokking_colab)
from sn_atomic_grokking_colab import AtomicConfig, run_atomic_sweep, calibrate_atomic_weight_decay

CONFIG = AtomicConfig(
    output_root='/kaggle/working/grokking_sn_atomic',
    protocol_name='atomic_native_v5_wd_search_f30',
    n_values=(6,), seeds=(42,),
    train_fraction=0.30,
    fixed_vocabulary=False,
    sampled_pairs_by_n={5:14_400, 6:518_400},
    d_model=128, d_mlp=512, d_head=32, n_heads=4,
    learning_rate=1e-3, weight_decay=0.2, betas=(0.9,0.98),
    max_steps=600_000, log_every=20, checkpoint_every=1_000,
    diagnostic_every=5_000,
    required_gap_steps=10_000,
    # Attach a prior Save-Version output as a Dataset to resume:
    resume_search_roots=('/kaggle/input',),
)
CALIBRATION = calibrate_atomic_weight_decay(CONFIG, weight_decays=(0.10,0.05,0.02,0.01))
display(CALIBRATION)

## Продолжение в следующей Kaggle-сессии

1. Нажмите Save Version → Save & Run All.
2. В новом notebook выберите Add Input → Your Work → output предыдущей версии.
3. Запустите этот notebook снова. Скрипт найдёт `checkpoint.pt` в `/kaggle/input`, скопирует его в writable `/kaggle/working` и продолжит обучение.

При возобновлении не меняйте `protocol_name`, seed или список weight decays. Каждый decay хранится в отдельной подпапке и подхватывает только собственный checkpoint.

In [ ]:
from pathlib import Path
import json, pandas as pd
for completed_path in sorted(Path(CONFIG.output_root).glob(CONFIG.protocol_name+'_wd*/S_6/seed_42/COMPLETED.json')):
    run_dir=completed_path.parent
    print(run_dir, json.loads(completed_path.read_text()))
    display(pd.read_csv(run_dir/'training_log.csv').tail(3))
print('Save as notebook output:', CONFIG.output_root)